# city evolution as radial growth

In [ ]:
import utca
import osmnx as ox
import networkx as nx
import pandas as pd
import geopandas as gpd
import shapely
from tqdm import tqdm
import matplotlib.pyplot as plt
import mpltern
from pathlib import Path

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    # Font
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Lines and markers
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    # Layout
    "figure.constrained_layout.use": True,
})

cm = 1 / 2.54

In [ ]:
admin_centres = pd.read_csv("data/admin_centres.csv")

In [ ]:
pop = utca.load_population()
towns = pop[pop[2011] > 100000].index.to_list()

# bp

In [ ]:
def get_bp_radial():
    lat, lon = 47.4978789, 19.0402383
    center = shapely.Point(lon, lat)
    G = ox.load_graphml('output/bp_simplified.graphml')
    G = utca.prepare_graph(G)
    streets = ox.graph_to_gdfs(G, nodes=False)
    radial = utca.get_radial_timeline(streets, center, 50, 500)
    return radial

In [ ]:
bp = get_bp_radial()

# main fig

In [ ]:
fig, ax = plt.subplots(figsize=(8*cm, 7*cm))
# budapest
ax.plot('n', 'v', '', alpha=0.6, data=bp, label='_') #trajectory
ax.scatter(x='n', y='v', data=bp.iloc[-1], label='Budapest') #end point
for varos in tqdm(towns):
    center = shapely.Point(admin_centres[admin_centres['name'] == varos].iloc[0][['lon', 'lat']].values)
    path_str = f'output/neat_20260103_171612/{varos}_simplified.graphml'
    G = ox.load_graphml(path_str)
    G = utca.prepare_graph(G)
    streets = ox.graph_to_gdfs(G, nodes=False)
    radial = utca.get_radial_timeline(streets, center, 50, 500)
    ax.plot('n', 'v', '', alpha=0.6, data=radial, label='_') #trajectory
    ax.scatter(x='n', y='v', data=radial.iloc[-1], label=varos) #end point
ax.legend()
ax.set_xlabel("$\\overline{n}^*$")
ax.set_ylabel("$\\overline{v}^*$")#, rotation=0)
#ax.set_title("Radial evolution of the largest cities")

In [ ]:
#fig.savefig("output/figs_maj10/radial_cities.pdf")

# kecskemet map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns


In [ ]:
observable10 = sns.color_palette([
    '#4269d0',
    '#efb118',
    '#ff725c',
    '#6cc5b0',
    '#3ca951',
    '#ff8ab7',
    '#a463f2',
    '#97bbf5',
    '#9c6b4e',
    '#9498a0',
]) #observable10

In [ ]:
def plot_map(varos, radius=None, cbar=True, stats_title=False):
    center = shapely.Point(admin_centres[admin_centres['name'] == varos].iloc[0][['lon', 'lat']].values)
    path_str = f'output/neat_20260103_171612/{varos}_simplified.graphml'
    G = ox.load_graphml(path_str)
    G = utca.prepare_graph(G)
    polygons = utca.poly_df(G)
    edges = ox.graph_to_gdfs(G, nodes=False)

    deg_col = "n_sides"
    min_deg = 2
    max_deg = 10   # 15 means "15+"
    gdf = polygons.copy()
    if radius:
        point_gdf = gpd.GeoDataFrame(geometry=[center], crs="EPSG:4326")
        point_gdf_proj = point_gdf.to_crs(utca.params.crs)
        buffer = point_gdf_proj.buffer(radius)
        gdf = gdf[gdf.intersects(buffer.iloc[0])]
        edges = edges[edges.intersects(buffer.iloc[0])]
    threshold = 100000
    gdf['threshold'] = gdf['area'] < threshold
    #gdf = gdf[gdf['area'] < threshold]
    gdf["deg_binned"] = gdf[deg_col].clip(upper=max_deg)
    n_classes = max_deg - min_deg + 1

    # HUSL palette (distinct categorical colors)
    #palette = sns.color_palette("husl", n_classes - 1)
    #palette = sns.mpl_palette('nipy_spectral_r', n_classes-1)
    palette = sns.color_palette(observable10, n_classes-1)

    # Add a neutral color for the overflow bin (15+)
    palette.append((0.5, 0.5, 0.5))  # dark gray

    cmap = mcolors.ListedColormap(palette)
    bounds = np.arange(min_deg, max_deg + 2)
    norm = mcolors.BoundaryNorm(bounds, cmap.N)
    fig, ax = plt.subplots(figsize=(7*cm, 7*cm))

    gdf[gdf['threshold']].plot(
        column="deg_binned",
        ax=ax,
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.2
    )
    gdf[~gdf['threshold']].plot(
        #column="deg_binned",
        ax=ax,
        #cmap=cmap,
        #norm=norm,
        color='0.9',
        edgecolor="black",
        linewidth=0.2
    )


    ax.set_aspect("equal")
    ax.axis("off")

    if stats_title:
        G2 = ox.graph_from_gdfs(*utca.rebuild_neat_graph(edges))
        _G = utca.prepare_graph(G2)
        # if not nx.is_connected(_G):
        largest_cc = max(nx.connected_components(_G), key=len)
        _G = _G.subgraph(largest_cc).copy()
        stats = utca.graph_stats(_G)
        ax.set_title(f"$\\overline{{n}}^* = {stats['n']:.2f}$, $\\overline{{v}}^* = {stats['v']:.2f}$")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    if cbar:
        cbar = fig.colorbar(
            sm,
            ax=ax,
            boundaries=bounds,
            ticks=np.arange(min_deg, max_deg + 1)+0.5,
            shrink=0.8,
            #orientation="horizontal"
        )

        ticklabels = [str(d) for d in range(min_deg, max_deg)]
        ticklabels.append(f"{max_deg}+")

        cbar.set_ticklabels(ticklabels)
        cbar.set_label("Polygon degree")

    return fig

In [ ]:
fig2 = plot_map('Nyíregyháza', 2500, stats_title=True)

In [ ]:
fig1 = plot_map('Nyíregyháza', 500, False, stats_title=True)

In [ ]:
#fig1.savefig('output/figs_maj10/radial_ex3.pdf', bbox_inches='tight')
#fig2.savefig('output/figs_maj10/radial_ex4.pdf', bbox_inches='tight')

In [ ]:
def plot_nodes(varos, save=False):
    center = shapely.Point(admin_centres[admin_centres['name'] == varos].iloc[0][['lon', 'lat']].values)
    path_str = f'output/neat_20260103_171612/{varos}_simplified.graphml'
    G = ox.load_graphml(path_str)
    G = utca.prepare_graph(G)
    polygons = utca.poly_df(G)
    nodes, edges = ox.graph_to_gdfs(G)
    nodes = nodes[nodes["n_corners"] > 1]
    column = "n_corners"
    n_classes = nodes[column].nunique()
    palette = sns.mpl_palette('Set1', n_classes)
    #palette = sns.hls_palette(n_classes)
    cmap = mcolors.ListedColormap(palette)
    fig, ax = plt.subplots(figsize=(8*cm, 7*cm))
    nodes.plot(ax=ax,column=column, cmap=cmap, categorical=True, markersize=0.05, legend=True, legend_kwds={'title': 'Degree'}, marker='o', edgecolor=None)
    edges.plot(ax=ax, color='black', alpha=0.5, linewidth=0.2)
    ax.set_axis_off()
    if save:
        fig.savefig("output/figs_jan2/radial_map.pdf")

In [ ]:
plot_nodes("Szombathely")

In [ ]:
def plot_joint(varos, save=False):
    fig, axs = plt.subplots(1, 2, figsize=(14*cm, 7*cm))
    path_str = f'output/neat_20260103_171612/{varos}_simplified.graphml'
    G = ox.load_graphml(path_str)
    G = utca.prepare_graph(G)
    polygons = utca.poly_df(G)
    nodes, edges = ox.graph_to_gdfs(G)
    nodes = nodes[nodes["n_corners"] > 1]
    column = "n_corners"
    n_classes = nodes[column].nunique()
    palette = sns.mpl_palette('Set1', n_classes)
    #palette = sns.hls_palette(n_classes)
    cmap = mcolors.ListedColormap(palette)
    
    nodes.plot(ax=axs[0],column=column, cmap=cmap, categorical=True, markersize=0.05, legend=True, legend_kwds={'title': 'Degree'}, marker='o', edgecolor=None)
    edges.plot(ax=axs[0], color='black', alpha=0.5, linewidth=0.2)
    axs[0].set_axis_off()

    deg_col = "n_sides"
    min_deg = 2
    max_deg = 15   # 15 means "15+"
    gdf = polygons.copy()
    gdf["deg_binned"] = gdf[deg_col].clip(upper=max_deg)
    n_classes = max_deg - min_deg + 1

    # HUSL palette (distinct categorical colors)
    #palette = sns.color_palette("husl", n_classes - 1)
    palette = sns.mpl_palette('gist_ncar', n_classes-1)

    # Add a neutral color for the overflow bin (15+)
    palette.append((0.5, 0.5, 0.5))  # dark gray

    cmap = mcolors.ListedColormap(palette)
    bounds = np.arange(min_deg, max_deg + 2)
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    gdf.plot(
        column="deg_binned",
        ax=axs[1],
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.2
    )

    axs[1].set_aspect("equal")
    axs[1].axis("off")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axs[1],
        boundaries=bounds,
        ticks=np.arange(min_deg, max_deg + 1)+0.5,
        shrink=0.8,
        #orientation="horizontal"
    )

    ticklabels = [str(d) for d in range(min_deg, max_deg)]
    ticklabels.append(f"{max_deg}+")

    cbar.set_ticklabels(ticklabels)
    cbar.set_label("Polygon degree")

    if save:
        fig.savefig("output/figs_jan2/radial_map2.png", dpi=300)

In [ ]:
plot_joint("Szombathely")